# AIP Petrol Prices (Terminal Gate Prices)

In [1]:
import io
import re
import urllib.request
from pathlib import Path

import pandas as pd
import mgplot as mg

In [2]:
CHART_DIR = "./CHARTS/AIP/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()
SHOW = False

In [3]:
# --- Fetch AIP Terminal Gate Price data (with caching) ---
# Note: AIP publishes the TGP file once a week on Fridays,
# typically before 10am AEDT. The file contains daily data
# for the week up to and including Friday. Good Friday
# published on the following Tuesday.

CACHE_DIR = Path("./CACHE/AIP")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
AIP_BASE = "https://www.aip.com.au/sites/default/files/download-files"
HEADERS = {"User-Agent": "Mozilla/5.0"}


def _candidate_fridays(n=4):
    """Yield (filename, folder) for the most recent n Fridays."""
    today = pd.Timestamp.today().normalize()
    days_since_friday = (today.weekday() - 4) % 7
    friday = today - pd.Timedelta(days=days_since_friday)
    for i in range(n):
        dt = friday - pd.Timedelta(weeks=i)
        filename = f"AIP_TGP_Data_{dt.strftime('%-d-%b-%Y')}.xlsx"
        folder = dt.strftime("%Y-%m")
        yield filename, folder


def get_tgp_data() -> bytes:
    """Get AIP TGP Excel: check cache then web for each recent Friday."""
    for filename, folder in _candidate_fridays():
        # Check cache first
        cache_path = CACHE_DIR / filename
        if cache_path.exists():
            print(f"Using cached: {filename}")
            return cache_path.read_bytes()

        # Try downloading
        url = f"{AIP_BASE}/{folder}/{filename}"
        req = urllib.request.Request(url, headers=HEADERS)
        try:
            data = urllib.request.urlopen(req, timeout=30).read()
            print(f"Downloaded: {filename}")
            (CACHE_DIR / filename).write_bytes(data)
            return data
        except urllib.error.HTTPError:
            continue

    raise FileNotFoundError("Could not find AIP TGP data file")


data = get_tgp_data()

petrol = pd.read_excel(
    io.BytesIO(data),
    sheet_name="Petrol TGP",
    header=0,
    index_col=0,
    parse_dates=True,
)
petrol.columns = petrol.columns.str.strip().str.replace("\n", " ")

diesel = pd.read_excel(
    io.BytesIO(data),
    sheet_name="Diesel TGP",
    header=0,
    index_col=0,
    parse_dates=True,
)
diesel.columns = diesel.columns.str.strip().str.replace("\n", " ")

print(f"Petrol: {petrol.index[0].date()} to {petrol.index[-1].date()}")
print(f"Diesel: {diesel.index[0].date()} to {diesel.index[-1].date()}")

Using cached: AIP_TGP_Data_27-Mar-2026.xlsx
Petrol: 2004-01-01 to 2026-03-27
Diesel: 2004-01-01 to 2026-03-27


In [4]:
# --- prepare to plot
START = pd.Period("2025-12-01", freq="D")

# Convert to PeriodIndex for mgplot
petrol.index = pd.DatetimeIndex(petrol.index).to_period("D")
diesel.index = pd.DatetimeIndex(diesel.index).to_period("D")

HORMUZ = {
    "x": pd.Period("2026-02-28", freq="D").ordinal,
    "color": "grey",
    "linestyle": "--",
    "linewidth": 1,
    "label": "Hormuz closure",
}
EXCISE = {
    "x": pd.Period("2026-04-01", freq="D").ordinal,
    "color": "grey",
    "linestyle": "-.",
    "linewidth": 1,
    "label": "Fuel Excise Reduction ~32c/l",
}
SOURCE = "Source: AIP"
DATA_TO = petrol.index[-1].strftime("%-d-%b-%Y")
FOOTER = f"Australia. Daily wholesale terminal gate prices (inc. GST), national average, business days. Data to {DATA_TO}."

fuel_prices = pd.DataFrame({
    "Petrol (ULP)": petrol["National Average"],
    "Diesel": diesel["National Average"],
}).dropna()

In [5]:
# --- Chart: Petrol and Diesel national average TGPs ---
def plot_fuel_prices():
    mg.multi_start(
        fuel_prices,
        function=mg.line_plot_finalise,
        starts=[0, START],
        title="Fuel Terminal Gate Prices: Petrol and Diesel",
        ylabel="Cents per litre",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        axvline=[HORMUZ, EXCISE],
        lfooter=FOOTER,
        rfooter=SOURCE,
        show=SHOW,
    )
    print(f"Chart saved to {CHART_DIR}")

plot_fuel_prices()

Chart saved to ./CHARTS/AIP/


In [6]:
# --- Inflation-adjust fuel prices using quarterly CPI ---
from abs_structured_capture import ReqsTuple, load_series


def get_daily_deflator(lookback_quarters: int = 2) -> tuple[pd.Series, str]:
    """Fetch quarterly CPI, interpolate to daily, and project forward to today.

    Returns a tuple of (deflator, dollar_label). The deflator series
    converts nominal prices to real prices in today's dollars.
    Projects CPI forward using the average quarterly growth rate
    over the lookback period.
    """

    cpi_req = ReqsTuple(
        "6401.0", "64010Appendix1a",
        "Index Numbers ;  All groups CPI, seasonally adjusted",
        "S", "", False, False, "",
    )
    cpi_q = load_series(cpi_req)

    # Project forward enough quarters to cover today
    recent_growth = cpi_q.pct_change().iloc[-lookback_quarters:].mean()
    today = pd.Timestamp.today().normalize()
    today_q = pd.Period(today, freq="Q")
    while cpi_q.index[-1] <= today_q:
        next_q = cpi_q.index[-1] + 1
        cpi_q[next_q] = cpi_q.iloc[-1] * (1 + recent_growth)

    # Interpolate to daily using end-of-quarter timestamps
    cpi_daily = (
        cpi_q.to_timestamp(how="end")
        .resample("D")
        .interpolate(method="linear")
    )

    # Rebase to today's dollars
    cpi_today = cpi_daily.loc[today]
    deflator = cpi_today / cpi_daily
    deflator.index = deflator.index.to_period("D")

    dollar_label = today.strftime("%B %Y") + " dollar terms"
    return deflator, dollar_label


def plot_real_fuel_prices():
    deflator, dollar_label = get_daily_deflator()
    real_fuel = pd.DataFrame({
        "Petrol (ULP)": fuel_prices["Petrol (ULP)"] * deflator,
        "Diesel": fuel_prices["Diesel"] * deflator,
    }).dropna()

    mg.line_plot_finalise(
        real_fuel,
        title="Real Fuel Terminal Gate Prices: Petrol and Diesel",
        ylabel="Cents per litre",
        xlabel=None,
        legend={"loc": "best", "fontsize": "x-small"},
        annotate=True,
        lfooter=f"Australia. Daily wholesale TGP (inc. GST). In {dollar_label}, using All Groups CPI (SA).",
        rfooter=f"{SOURCE}, ABS",
        show=SHOW,
    )
    print(f"Chart saved to {CHART_DIR}")

plot_real_fuel_prices()

Chart saved to ./CHARTS/AIP/


In [7]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda

Last updated: 2026-04-07 09:49:03

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.12.0

conda environment: n/a

Compiler    : Clang 20.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot : 0.2.21
pandas : 3.0.2
pathlib: 1.0.1
re     : 2.2.1

Watermark: 2.6.0



In [8]:
print("Finished")

Finished
